# Project 6 — Elliptic PDE: 2-D Laplace Equation
**Topic:** Solving $\nabla^2 U = 0$ on a 2-D domain by successive over-relaxation (SOR). Physical application: steady-state heat conduction or electrostatic potential.


---
## 1. Set Up

### Physical Motivation
The **2-D Laplace equation** $\nabla^2 U = 0$ describes:
- **Electrostatics**: the potential $\phi$ in a charge-free region.
- **Steady heat flow**: temperature $T$ when $\partial T/\partial t = 0$ and no heat sources.
- **Fluid mechanics**: the velocity potential for irrotational incompressible flow.

### Domain and Boundary Conditions
Unit square $[0,1]\times[0,1]$ with **non-homogeneous Dirichlet BC** on the top edge:
$$U(x, 1) = 100\sin(\pi x), \quad U(x,0) = U(0,y) = U(1,y) = 0$$

### Analytic Solution (Separation of Variables)
The exact solution is:
$$U^*(x,y) = 100\sin(\pi x)\frac{\sinh(\pi y)}{\sinh(\pi)}$$

This validates the code precisely. **Physical meaning**: the sine-shaped heat source on the top edge decays exponentially as you move away from it — $\sinh(\pi y)/\sinh(\pi) \approx e^{\pi(y-1)}$ for small $y$.

### Finite Difference Discretisation
The 5-point stencil for $\nabla^2 U \approx 0$:
$$U_{i,j} \approx \frac{1}{4}(U_{i-1,j} + U_{i+1,j} + U_{i,j-1} + U_{i,j+1})$$

Each interior point equals the **average of its four neighbours** — this is the **discrete mean value property** of harmonic functions.

### ⚠️ Critical Performance Warning
The `solve_laplace_sor` function uses **nested Python loops** over all interior grid points. For an $81\times81$ grid: $79^2 \approx 6241$ iterations **per SOR sweep**, and $\sim 1000$ sweeps needed. That is $6 \times 10^6$ Python loop iterations. This is **very slow** — vectorised NumPy or C would be $100\times$ faster.


In [ ]:
import os, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

OUT = Path("output_elliptic_laplace")
OUT.mkdir(exist_ok=True)
np.set_printoptions(precision=6, suppress=True)
print(f"Output directory: {OUT.resolve()}")


---
## 2. Functions

### `initialize_plate()`

```python
U[-1, :] = 100.0 * np.sin(np.pi * x)  # top boundary
U[0, :] = 0.0; U[:, 0] = 0.0; U[:, -1] = 0.0
```
Sets boundary conditions. **Array indexing convention**: `U[j, i]` where `j` is the row (y-direction, 0 = bottom, -1 = top) and `i` is the column (x-direction). This follows the mathematical convention $U(x,y)$ but **transposes** the matrix — a frequent source of confusion.

### `laplace_residual(U)`
```python
R = U[1:-1, 1:-1] - 0.25*(U[1:-1, :-2] + U[1:-1, 2:] + U[:-2, 1:-1] + U[2:, 1:-1])
```
**Vectorised residual**: computes $U_{i,j} - \frac{1}{4}(U_{i\pm1,j} + U_{i,j\pm1})$ for all interior points simultaneously. Uses NumPy array slicing — no Python loop needed. This is much faster than the SOR update loop.

### `solve_laplace_sor(U, omega)`

```python
average_neighbors = 0.25*(U[j,i-1] + U[j,i+1] + U[j-1,i] + U[j+1,i])
U[j,i] = (1.0 - omega)*U[j,i] + omega*average_neighbors
```
**SOR update**: blend between old value (weight $1-\omega$) and the local average of neighbours (weight $\omega$). The sequential update ensures that once `U[j,i-1]` and `U[j-1,i]` are updated, their new values are used immediately — the same Gauss-Seidel principle as Project 05.

**Why $\omega = 1.8$?** The optimal SOR parameter for the Laplace equation on an $n\times n$ grid is:
$$\omega_{\text{opt}} = \frac{2}{1 + \sin(\pi/(n-1))}$$

For $n=81$: $\omega_{\text{opt}} \approx 1.924$. The code uses $\omega = 1.85$ — close to optimal but slightly under, which is **safer**: over-estimating $\omega$ causes divergence, under-estimating causes slower convergence.

### `analytic_solution(x, y)`
```python
return top_amplitude*np.sin(np.pi*X)*np.sinh(np.pi*Y)/np.sinh(np.pi)
```
Exact closed-form solution. `np.meshgrid(x, y)` creates 2-D arrays `X[j,i]=x[i]`, `Y[j,i]=y[j]` — consistent with the `U[j,i]` storage convention. `np.sinh` is the hyperbolic sine — safe for $\pi y \leq \pi$ (no overflow).

### 🔧 Improvement Directions
1. **Vectorised Red-Black SOR**: split interior points into "red" and "black" squares (like a checkerboard). Update all red points (which depend only on black) simultaneously, then all black. This vectorises the inner loop.
2. **Multigrid**: V-cycle with restriction and prolongation operators. Converges in $\mathcal{O}(n^2)$ total work — optimal.
3. **FFT solver**: for this specific geometry and BC, the discrete sine transform gives the exact solution in $\mathcal{O}(n^2\log n)$.
4. **Non-homogeneous BCs**: adding a source term $f(x,y)$ gives the Poisson equation — same method, different RHS.

### ⚠️ Known Weaknesses
- **Nested Python loop**: for production use, implement in C/Fortran or use `scipy.sparse.linalg.spsolve`.
- **Square domain only**: extending to irregular geometries requires finite elements or unstructured meshes.
- **Convergence criterion**: `laplace_residual` uses $L^\infty$ norm — for nearly-converged solutions, a single outlier point can keep the iteration going unnecessarily.


In [ ]:
def initialize_plate(nx=81, ny=81):
    x = np.linspace(0.0, 1.0, nx)
    y = np.linspace(0.0, 1.0, ny)
    U = np.zeros((ny, nx), dtype=float)        # U[j,i]: row=y, col=x
    U[-1, :] = 100.0 * np.sin(np.pi * x)      # top BC: U(x,1) = 100 sin(pi x)
    U[0,  :] = 0.0; U[:, 0] = 0.0; U[:, -1] = 0.0  # other edges: zero
    return x, y, U

def laplace_residual(U):
    R = U[1:-1, 1:-1] - 0.25*(
        U[1:-1, :-2] + U[1:-1, 2:] + U[:-2, 1:-1] + U[2:, 1:-1])
    return np.max(np.abs(R))

def solve_laplace_sor(U, omega=1.8, tol=1e-6, max_iter=20000):
    U = U.copy()
    residuals = []
    ny, nx = U.shape
    for it in range(max_iter):
        for j in range(1, ny-1):
            for i in range(1, nx-1):
                avg = 0.25*(U[j,i-1] + U[j,i+1] + U[j-1,i] + U[j+1,i])
                U[j,i] = (1.0 - omega)*U[j,i] + omega*avg
        res = laplace_residual(U)
        residuals.append(res)
        if res < tol: break
    return U, np.array(residuals)

def analytic_solution(x, y, top_amplitude=100.0):
    X, Y = np.meshgrid(x, y)
    return top_amplitude * np.sin(np.pi*X) * np.sinh(np.pi*Y) / np.sinh(np.pi)


---
## 3, 4, 5. Calculation, Output & Visualisation

### What the 2-D Plot Shows
- **Contour plot**: lines of constant $U$ — these are equipotential lines (electrostatics) or isotherms (heat). They are perpendicular to the gradient (heat flux / electric field).
- **Heat map**: colour encodes $U(x,y)$. The sine-shaped hot top boundary creates a smooth distribution that decays toward zero on the three cold edges.
- **3-D surface**: shows $U$ as a height field — useful for seeing the smooth harmonic nature of the solution (no local maxima or minima in the interior — this is the **maximum principle** for harmonic functions).


In [ ]:
NX, NY = 81, 81
OMEGA   = 1.85
TOL     = 1e-6

x, y, U0 = initialize_plate(nx=NX, ny=NY)
print("Running SOR... (this may take ~30 seconds in Python loops)")
U_sor, residuals = solve_laplace_sor(U0, omega=OMEGA, tol=TOL)
U_exact = analytic_solution(x, y)

max_err = np.max(np.abs(U_sor - U_exact))
print(f"SOR converged in {len(residuals)} iterations")
print(f"Final residual: {residuals[-1]:.2e}")
print(f"Max error vs. analytic: {max_err:.4f}")

fig = plt.figure(figsize=(16, 5))
X, Y = np.meshgrid(x, y)

# Heatmap
ax1 = fig.add_subplot(131)
im = ax1.contourf(X, Y, U_sor, levels=20, cmap='hot')
ax1.contour(X, Y, U_sor, levels=20, colors='k', linewidths=0.4, alpha=0.5)
plt.colorbar(im, ax=ax1)
ax1.set_title('SOR solution — heatmap'); ax1.set_xlabel('x'); ax1.set_ylabel('y')

# Error map
ax2 = fig.add_subplot(132)
err_map = np.abs(U_sor - U_exact)
im2 = ax2.contourf(X, Y, err_map, levels=20, cmap='viridis')
plt.colorbar(im2, ax=ax2)
ax2.set_title(f'|Error| (max={max_err:.3f})')
ax2.set_xlabel('x'); ax2.set_ylabel('y')

# 3D surface
ax3 = fig.add_subplot(133, projection='3d')
ax3.plot_surface(X, Y, U_sor, cmap='hot', alpha=0.9, linewidth=0)
ax3.set_title('3D surface'); ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_zlabel('U')

plt.tight_layout()
plt.savefig(OUT / "laplace_2d.png", dpi=130, bbox_inches='tight')
plt.show()

# Convergence
plt.figure(figsize=(7,4))
plt.semilogy(residuals, 'steelblue', lw=1.5)
plt.xlabel('SOR iteration'); plt.ylabel('Max residual')
plt.title(f'SOR convergence (ω={OMEGA})')
plt.grid(True, which='both', alpha=0.4)
plt.savefig(OUT / "laplace_convergence.png", dpi=130, bbox_inches='tight')
plt.show()

print("\n" + "=" * 55)
print("SANITY CHECKS — Project 06 Elliptic Laplace")
print("=" * 55)
ok1 = max_err < 0.5
print(f"  Max error < 0.5: {max_err:.4f}  {'✓' if ok1 else '✗'}")
ok2 = np.all(U_sor[0,:] < 0.1) and np.all(U_sor[:,0] < 0.1)
print(f"  BC: bottom & left ≈ 0:  {'✓' if ok2 else '✗'}")
ok3 = np.max(U_sor) < 101 and np.min(U_sor) > -0.1
print(f"  Maximum principle: 0 ≤ U ≤ 100:  {'✓' if ok3 else '✗'}")
ok4 = np.max(np.abs(U_sor[-1,:] - 100.0*np.sin(np.pi*x))) < 0.01
print(f"  Top BC satisfied:  {'✓' if ok4 else '✗'}")
print("=" * 55)
